# Forest Plot Extraction — Eval

Reads `table_catalog.csv`, filters for `table_type=forest_plot` and `status=complete`, runs extraction on each entry, compares to ground truth, and prints accuracy (cell_accuracy + row_match).

In [9]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from shared.client import client, DEFAULT_MODEL
from shared.pdf import render_pages
from shared.eval import compare_tables, print_accuracy_summary
from agents.forest_plot.extract import extract_forest_plots_from_page, stitch_forest_plot_results

In [10]:
MODEL = DEFAULT_MODEL

catalog = pd.read_csv("../table_catalog.csv")
catalog["status"] = catalog["status"].str.strip()
forest_entries = catalog[catalog["table_type"] == "forest_plot"].copy()
# Only eval entries with status=complete (ground truth available)
eval_entries = forest_entries[forest_entries["status"] == "complete"]
print(f"Forest plot entries to evaluate: {len(eval_entries)}")
eval_entries

Forest plot entries to evaluate: 2


,paper_id,page,figure_id,table_type,variation,description,ground_truth_path,cell_accuracy,row_match,notes,status
0,ctt_lancet_2024,7,fig1,forest_plot,subgroups with interaction p-values,Statin vs placebo new-onset diabetes by intensity,papers/ctt_lancet_2024/ground_truth/p7_fig1_fo...,100.0%,13.0,NaN,complete
1,ctt_lancet_2024,9,fig1,forest_plot,subgroups with interaction p-values,Statin vs placebo new-onset diabetes by intensity,papers/ctt_lancet_2024/ground_truth/p9_fig3_fo...,NaN,NaN,NaN,complete


In [11]:
results = []

for _, row in eval_entries.iterrows():
    paper_id = row["paper_id"]
    page = int(row["page"])
    pdf_path = os.path.join("..", "papers", paper_id, "paper.pdf")
    gt_path = os.path.join("..", row["ground_truth_path"])

    print(f"\n--- {paper_id} p{page} ({row['description']}) ---")

    # Render and extract
    images = render_pages(pdf_path, [page])
    page_result = extract_forest_plots_from_page(client, MODEL, images[page])
    page_results = {page: page_result}
    dfs = stitch_forest_plot_results(page_results)

    if not dfs:
        print("  No forest plots extracted!")
        continue

    # Compare first extracted plot to ground truth
    _, df_extracted = dfs[0]
    df_truth = pd.read_csv(gt_path)
    result = compare_tables(df_truth, df_extracted)
    result["paper_id"] = paper_id
    result["page"] = page
    results.append(result)
    print_accuracy_summary(result)


--- ctt_lancet_2024 p7 (Statin vs placebo new-onset diabetes by intensity) ---
ACCURACY SUMMARY
Ground truth rows:  13
Extracted rows:     13
Rows compared:      13
Missing rows:       0
Extra rows:         0

Total cells compared: 78
Correct cells:        78
Cell accuracy:        100.0%
Row match:            100.0%  (13/13 rows fully correct)

Perfect match! No mismatches found.

PER-COLUMN ACCURACY


,Column,Total Cells,Correct,Errors,Accuracy
0,unnamed: 0,13,13,0,100.0%
1,events (% per annum),13,13,0,100.0%
2,unnamed: 2,13,13,0,100.0%
3,observed-expected,13,13,0,100.0%
4,unnamed: 4,13,13,0,100.0%
5,rate ratio (ci),13,13,0,100.0%



--- ctt_lancet_2024 p9 (Statin vs placebo new-onset diabetes by intensity) ---
ACCURACY SUMMARY
Ground truth rows:  11
Extracted rows:     11
Rows compared:      11
Missing rows:       0
Extra rows:         0

Total cells compared: 66
Correct cells:        66
Cell accuracy:        100.0%
Row match:            100.0%  (11/11 rows fully correct)

Perfect match! No mismatches found.

PER-COLUMN ACCURACY


,Column,Total Cells,Correct,Errors,Accuracy
0,unnamed: 0,11,11,0,100.0%
1,events (% per annum),11,11,0,100.0%
2,unnamed: 2,11,11,0,100.0%
3,observed-expected,11,11,0,100.0%
4,unnamed: 4,11,11,0,100.0%
5,rate ratio (ci),11,11,0,100.0%


In [12]:
# Aggregate accuracy
if results:
    total_cells = sum(r["total_cells"] for r in results)
    correct_cells = sum(r["correct_cells"] for r in results)
    total_rows = sum(r["rows_compared"] for r in results)
    rows_correct = sum(r["rows_fully_correct"] for r in results)
    print(f"\n{'=' * 60}")
    print(f"AGGREGATE CELL ACCURACY: {correct_cells}/{total_cells} = {correct_cells/total_cells:.1%}")
    print(f"AGGREGATE ROW MATCH:     {rows_correct}/{total_rows} = {rows_correct/total_rows:.1%}")
    print(f"{'=' * 60}")
else:
    print("No entries evaluated.")


AGGREGATE CELL ACCURACY: 144/144 = 100.0%
AGGREGATE ROW MATCH:     24/24 = 100.0%
